# 02 Training and Logs

This notebook documents how the three PPO agents were trained and how TensorBoard logs were stored.

It demonstrates:
- importing the training pipeline
- locating trained models
- locating TensorBoard event files
- optionally running a very short demo training loop

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))

print("Project root:", ROOT)

Project root: /Users/joel_mathew/f1-rl-safety


In [2]:
from stable_baselines3 import PPO
from f1_rl_safety.f1_env import F1RaceEnv

## Locate models and TensorBoard logs

The project stores final trained PPO policies in `models/` and TensorBoard logs in `logs/`.
TensorBoard recursively searches the log directory for `tfevents` files.

In [3]:
models_dir = ROOT / "models"
logs_dir = ROOT / "logs"
archive_dir = ROOT / "archive"

print("Models:")
if models_dir.exists():
    for f in sorted(models_dir.iterdir()):
        print("-", f.name)
else:
    print("[missing]")

print("\nTensorBoard event files:")
if logs_dir.exists():
    tb_files = list(logs_dir.rglob("*tfevents*"))
    for f in tb_files:
        print("-", f.relative_to(ROOT))
else:
    print("[missing]")

print("\nArchive:")
if archive_dir.exists():
    for f in sorted(archive_dir.iterdir()):
        print("-", f.name)
else:
    print("[missing]")

Models:
- ppo_rulebook.zip
- ppo_safe.zip
- ppo_unconstrained.zip

TensorBoard event files:
- logs/rulebook/PPO_1/events.out.tfevents.1781381332.Joels-MacBook-Pro.local.42473.1
- logs/unconstrained/PPO_1/events.out.tfevents.1781381255.Joels-MacBook-Pro.local.42473.0
- logs/safe/PPO_1/events.out.tfevents.1781381408.Joels-MacBook-Pro.local.42473.2

Archive:
- logs_old_20260613_204626
- logs_old_20260613_210650
- models_old_20260613_204626
- models_old_20260613_210650


## Example of how PPO training is configured

This section is a lightweight, readable version of the training workflow.
It is not intended to replace `src/f1_rl_safety/train.py`, which remains the main implementation.

In [8]:
env = F1RaceEnv()

demo_model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    tensorboard_log=None,
)
print("Demo model created. TensorBoard logging is disabled for this notebook run")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Demo model created. TensorBoard logging is disabled for this notebook run


## Optional: run a short demo training session

This cell is intentionally short so the notebook stays runnable.
It does **not** reproduce the full experiment, which would take much longer and is already represented by the saved models and logs.

In [ ]:
RUN_DEMO_TRAINING = False

if RUN_DEMO_TRAINING:
    demo_model.learn(total_timesteps=1000)
    demo_path = ROOT / "models" / "ppo_demo_notebook"
    demo_model.save(str(demo_path))
    print("Saved demo model to:", demo_path.with_suffix(".zip"))
else:
    print("Demo training skipped. Set RUN_DEMO_TRAINING = True to run it.")

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 36.5     |
|    ep_rew_mean     | -49.1    |
| time/              |          |
|    fps             | 7628     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
Saved demo model to: /Users/joel_mathew/f1-rl-safety/models/ppo_demo_notebook.zip


## How to open TensorBoard logs

From the project root in a VS Code terminal or macOS Terminal, run:

```bash
tensorboard --logdir logs --port 6006
```

Then open the local URL shown in the terminal, usually:

`http://localhost:6006`

TensorBoard will recursively search `logs/` for event files.

In [10]:
tb_files = list((ROOT / "logs").rglob("*tfevents*")) if (ROOT / "logs").exists() else []
print("Found", len(tb_files), "TensorBoard event files.")
for f in tb_files:
    print("-", f.relative_to(ROOT))

Found 3 TensorBoard event files.
- logs/rulebook/PPO_1/events.out.tfevents.1781381332.Joels-MacBook-Pro.local.42473.1
- logs/unconstrained/PPO_1/events.out.tfevents.1781381255.Joels-MacBook-Pro.local.42473.0
- logs/safe/PPO_1/events.out.tfevents.1781381408.Joels-MacBook-Pro.local.42473.2


## Notes

This notebook exists to show:
- how training was organised,
- where the log directories are,
- and where final models are stored.

The full training script remains in:
- `src/f1_rl_safety/train.py`